<a href="https://colab.research.google.com/github/SKM1919/liftlens/blob/main/notebooks/02_uplift_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LiftLens: Uplift Models (Medium tier)
Goal: predict the email's effect on each individual customer, not just on average.

In [1]:
!pip install scikit-uplift xgboost -q

import pandas as pd
import numpy as np
from sklift.datasets import fetch_hillstrom
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# Load data
bunch = fetch_hillstrom(target_col='all')
df = pd.concat([bunch.data, bunch.target, bunch.treatment], axis=1)

# Keep only Mens E-Mail vs No E-Mail, and make a 0/1 treatment flag
df = df[df['segment'].isin(['Mens E-Mail', 'No E-Mail'])].copy()
df['treatment'] = (df['segment'] == 'Mens E-Mail').astype(int)

# Features: customer info known BEFORE the email was sent
features = ['recency', 'history', 'mens', 'womens', 'zip_code', 'newbie', 'channel']
X = pd.get_dummies(df[features], columns=['zip_code', 'channel'], dtype=int)
y = df['visit']
t = df['treatment']

# 70% train, 30% test, keeping treatment/visit balance the same in both
X_train, X_test, y_train, y_test, t_train, t_test = train_test_split(
    X, y, t, test_size=0.3, random_state=42,
    stratify=t.astype(str) + y.astype(str)
)

print("Train:", X_train.shape, " Test:", X_test.shape)
X_train.head()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.5 MB/s eta 0:00:00


Hillstrom dataset:   0%|          | 0.00/443k [00:00<?, ?iB/s]

Train: (29829, 11)  Test: (12784, 11)


,recency,history,mens,womens,newbie,zip_code_Rural,zip_code_Surburban,zip_code_Urban,channel_Multichannel,channel_Phone,channel_Web
37645,2,219.05,0,1,1,0,0,1,1,0,0
56937,6,206.61,0,1,0,0,1,0,0,0,1
19588,2,600.39,0,1,1,0,0,1,0,1,0
14247,2,387.24,1,0,0,0,0,1,1,0,0
48850,4,29.99,1,0,1,0,1,0,0,0,1


In [2]:
params = dict(n_estimators=200, max_depth=3, learning_rate=0.05,
              random_state=42, eval_metric='logloss')

# Model A: learns from emailed customers only
model_treated = XGBClassifier(**params)
model_treated.fit(X_train[t_train == 1], y_train[t_train == 1])

# Model B: learns from non-emailed customers only
model_control = XGBClassifier(**params)
model_control.fit(X_train[t_train == 0], y_train[t_train == 0])

# For every test customer, ask both models and subtract
p_if_emailed = model_treated.predict_proba(X_test)[:, 1]
p_if_not = model_control.predict_proba(X_test)[:, 1]
uplift_t = p_if_emailed - p_if_not

print("Done. First 5 predicted uplifts:", np.round(uplift_t[:5], 3))

Done. First 5 predicted uplifts: [0.116 0.052 0.064 0.081 0.014]


In [3]:
actual_lift = y_test[t_test == 1].mean() - y_test[t_test == 0].mean()
print(f"Actual lift on test set:   {actual_lift * 100:.2f} pp")
print(f"Average predicted uplift:  {uplift_t.mean() * 100:.2f} pp")
print(f"Share with negative uplift: {(uplift_t < 0).mean() * 100:.1f}%")

pd.Series(uplift_t * 100).describe().round(2)


Actual lift on test set:   7.65 pp
Average predicted uplift:  7.62 pp
Share with negative uplift: 1.7%


,0
count,12784.00
mean,7.62
std,3.88
min,-14.48
25%,5.34
50%,7.11
75%,9.55
max,26.85


## S-learner and model comparison

In [4]:
# Add the treatment flag as a feature
X_train_s = X_train.copy()
X_train_s['treatment'] = t_train.values

model_s = XGBClassifier(**params)
model_s.fit(X_train_s, y_train)

# Ask the same model two "what if" questions
X_test_emailed = X_test.copy()
X_test_emailed['treatment'] = 1

X_test_not = X_test.copy()
X_test_not['treatment'] = 0

uplift_s = (model_s.predict_proba(X_test_emailed)[:, 1]
            - model_s.predict_proba(X_test_not)[:, 1])

print(f"Average predicted uplift (S): {uplift_s.mean() * 100:.2f} pp")
print(f"Share with negative uplift (S): {(uplift_s < 0).mean() * 100:.1f}%")

Average predicted uplift (S): 7.43 pp
Share with negative uplift (S): 0.1%


In [5]:
comparison = pd.DataFrame({
    'T-learner': uplift_t * 100,
    'S-learner': uplift_s * 100
})

print(comparison.describe().round(2))
print("\nCorrelation between the two:", round(comparison.corr().iloc[0, 1], 2))

       T-learner  S-learner
count   12784.00   12784.00
mean        7.62       7.43
std         3.88       2.79
min       -14.48      -2.78
25%         5.34       5.65
50%         7.11       7.16
75%         9.55       8.41
max        26.85      22.87

Correlation between the two: 0.66
